# Основная модель: CatBoost (Pool + тюнинг)
Цель — получить сильную базовую модель и аккуратно подобрать гиперпараметры.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "README.md").exists()
    and (path / "Data_making").is_dir()
    and (path / "Models").is_dir()
)
DATA_PATH = (
    PROJECT_ROOT
    / "Data_making"
    / "synthetic_education_dushanbe_WITH_ROUNDED.csv"
)
METRICS_DIR = PROJECT_ROOT / "Models" / "Compare models"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
from catboost import CatBoostClassifier, Pool, cv
from sklearn.model_selection import train_test_split, ParameterGrid

In [2]:
df = pd.read_csv(DATA_PATH)
df.head()

,ID_ученика,Класс,Район,Средний_балл,Часы_самоподготовки_в_неделю,Посещаемость_%,Уверенность_в_себе,Уровень_стресса_перед_экзаменом,Пропущенные_дни,Тип_школы,Рез_экзамена,Индекс_качества_школы,Стабильность_преподавателей,Доступ_к_ресурсам,Образовательная_среда
0,1,11,Исмоили Сомони,3.883,15,81.091,4.0,6.0,8,Обычная школа,1,0.4000,0.2911,0.3107,0.1970
1,2,11,Сино,3.915,15,87.872,4.0,4.0,10,Обычная школа,0,0.4443,0.2559,0.4509,0.5341
2,3,11,Шохмансур,3.660,5,71.282,5.0,6.0,13,Обычная школа,0,0.2499,0.1305,0.1049,0.2666
3,4,11,Исмоили Сомони,4.401,15,84.693,5.0,5.0,10,Обычная школа,1,0.5137,0.5229,0.4238,0.4449
4,5,11,Сино,4.311,12,75.941,5.0,4.0,9,Лицей/гимназия,1,0.4473,0.4858,0.4935,0.4143


In [3]:
# Общий набор признаков для честного сравнения моделей
TARGET_COL = "Рез_экзамена"
ID_COL = "ID_ученика"
LEAKAGE_RISK_COLS = ["Средний_балл"]
FEATURE_COLUMNS = [
    "Класс",
    "Район",
    "Часы_самоподготовки_в_неделю",
    "Посещаемость_%",
    "Уверенность_в_себе",
    "Уровень_стресса_перед_экзаменом",
    "Пропущенные_дни",
    "Тип_школы",
    "Индекс_качества_школы",
    "Стабильность_преподавателей",
    "Доступ_к_ресурсам",
    "Образовательная_среда",
]
y = df[TARGET_COL]
X = df[FEATURE_COLUMNS].copy()

# Проверка бинарности и баланса
unique_vals = np.sort(y.unique())
print('Target unique:', unique_vals)
print('Class balance:\n', y.value_counts(normalize=True))

if len(unique_vals) != 2:
    raise ValueError('Ожидается бинарный таргет для CatBoostClassifier.')

Target unique: [0 1]
Class balance:
 Рез_экзамена
1    0.622295
0    0.377705
Name: proportion, dtype: float64


In [4]:
# Категориальные признаки
categorical_features = X.select_dtypes(include=['object']).columns
cat_feature_indices = [X.columns.get_loc(c) for c in categorical_features]

# Единое стратифицированное разделение: 60% train / 20% validation / 20% test
RANDOM_STATE = 42
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid,
    y_train_valid,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_train_valid,
)

# Class weights рассчитываются только по train.
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
class_weights = [1.0, float(pos_weight)]

train_pool = Pool(X_train, y_train, cat_features=cat_feature_indices)
valid_pool = Pool(X_valid, y_valid, cat_features=cat_feature_indices)
test_pool = Pool(X_test, y_test, cat_features=cat_feature_indices)

print('Split sizes:', len(X_train), len(X_valid), len(X_test))

Split sizes: 915 305 305


## Базовый CatBoost
Стартовая модель для ориентира перед тюнингом.


In [5]:
from sklearn.metrics import (
    accuracy_score, average_precision_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, classification_report, confusion_matrix, roc_curve
)

In [6]:
base_model = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=RANDOM_STATE,
    task_type='CPU',
    iterations=500,
    class_weights=class_weights,
    verbose=False
)

base_model.fit(train_pool, eval_set=valid_pool, use_best_model=True, plot=False)
base_pred = base_model.predict_proba(X_valid)[:, 1]

print('Base AUC:', roc_auc_score(y_valid, base_pred))



# Предсказания вероятностей и классов
base_pred_proba = base_model.predict_proba(X_valid)[:, 1]
base_pred_class = (base_pred_proba >= 0.5).astype(int)

# Метрики
print("Base Accuracy:", accuracy_score(y_valid, base_pred_class))
print("Base Precision:", precision_score(y_valid, base_pred_class))
print("Base Recall:", recall_score(y_valid, base_pred_class))
print("Base F1:", f1_score(y_valid, base_pred_class))
print("Base ROC-AUC:", roc_auc_score(y_valid, base_pred_proba))
print("Base LogLoss:", log_loss(y_valid, base_pred_proba))

# Дополнительно: подробный отчёт по классам
print("\nClassification Report:\n", classification_report(y_valid, base_pred_class))

Base AUC: 0.7699313501144165
Base Accuracy: 0.6327868852459017
Base Precision: 0.7708333333333334
Base Recall: 0.5842105263157895
Base F1: 0.6646706586826348
Base ROC-AUC: 0.7699313501144165
Base LogLoss: 0.5945826709791763

Classification Report:
               precision    recall  f1-score   support

           0       0.51      0.71      0.59       115
           1       0.77      0.58      0.66       190

    accuracy                           0.63       305
   macro avg       0.64      0.65      0.63       305
weighted avg       0.67      0.63      0.64       305



In [7]:
# Важность признаков базовой модели (для анализа)
base_importance = base_model.get_feature_importance(type='FeatureImportance')
base_fi_df = (
    pd.DataFrame({'feature': X.columns, 'importance': base_importance})
      .sort_values('importance', ascending=False)
      .reset_index(drop=True)
)

# Optional extended feature experiment (не используется в основном benchmark):
# df_fe['Бал_на_стресс'] = ...
# df_fe['Бал_на_уверенность'] = ...
# df_fe['Балл_минус_стресс'] = ...
# df_fe['Пропуски_на_стресс'] = ...
# df_fe['Среда_школы_среднее'] = ...
# df_fe['Качество_на_среду'] = ...
# df_fe['Качество_на_ресурсы'] = ...

# Основной benchmark использует только общий набор из 12 исходных признаков.
# Train/validation/test и Pool-объекты созданы один раз выше и не пересоздаются.


## Тюнинг гиперпараметров
Используем `catboost.cv` (5‑fold) с early stopping. Это медленнее, но надежнее.


In [8]:
base_params = {
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': RANDOM_STATE,
    'task_type': 'CPU',
    'verbose': False,
    'od_type': 'Iter',
    'od_wait': 50,
    'class_weights': class_weights
}

param_grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.03, 0.1, 0.2],
    'l2_leaf_reg': [1, 3, 5, 7],
    'iterations': [400, 800, 1200],
    'bagging_temperature': [0, 1, 3],
    'random_strength': [1, 2, 5]
}

In [9]:
# Сколько комбинаций реально прогнать
max_evals = 20
grid = list(ParameterGrid(param_grid))
rng = np.random.default_rng(RANDOM_STATE)
rng.shuffle(grid)
grid = grid[:max_evals]

best_auc = -1
best_params = None

for params in grid:
    cv_params = {**base_params, **params}
    cv_data = cv(
        pool=train_pool,
        params=cv_params,
        fold_count=5,
        stratified=True,
        partition_random_seed=RANDOM_STATE,
        shuffle=True,
        early_stopping_rounds=50,
        verbose=False
    )
    auc = cv_data['test-AUC-mean'].max()

    if auc > best_auc:
        best_auc = auc
        best_params = params

print('Best CV AUC:', best_auc)
print('Best params:', best_params)

Training on fold [0/5]

bestTest = 0.7535087719
bestIteration = 20

Training on fold [1/5]

bestTest = 0.7876938724
bestIteration = 73

Training on fold [2/5]

bestTest = 0.8254513094
bestIteration = 20

Training on fold [3/5]

bestTest = 0.7506356471
bestIteration = 10

Training on fold [4/5]

bestTest = 0.7064255483
bestIteration = 19

Training on fold [0/5]

bestTest = 0.7368421053
bestIteration = 17

Training on fold [1/5]

bestTest = 0.7838799898
bestIteration = 87

Training on fold [2/5]

bestTest = 0.8198576151
bestIteration = 32

Training on fold [3/5]

bestTest = 0.7358250699
bestIteration = 41

Training on fold [4/5]

bestTest = 0.7149544697
bestIteration = 4

Training on fold [0/5]

bestTest = 0.7364661654
bestIteration = 15

Training on fold [1/5]

bestTest = 0.7843885075
bestIteration = 103

Training on fold [2/5]

bestTest = 0.8232901093
bestIteration = 47

Training on fold [3/5]

bestTest = 0.7337274345
bestIteration = 18

Training on fold [4/5]

bestTest = 0.7001410799


## Финальная модель и метрики
Validation используется для early stopping и threshold selection. Финальные метрики считаются только на test.


In [10]:
if best_params is None:
    best_params = {}

final_model = CatBoostClassifier(
    **base_params,
    **best_params
)

final_model.fit(train_pool, eval_set=valid_pool, use_best_model=True)
validation_proba = final_model.predict_proba(valid_pool)[:, 1]

# Подбор порога: максимизируем Recall при ограничении на Precision
TARGET_MIN_PRECISION = 0.80
threshold_grid = np.linspace(0.1, 0.9, 161)

threshold_candidates = []
for thr in threshold_grid:
    pred_thr = (validation_proba >= thr).astype(int)
    p = precision_score(y_valid, pred_thr, zero_division=0)
    r = recall_score(y_valid, pred_thr, zero_division=0)
    threshold_candidates.append({
        'threshold': float(thr),
        'precision': p,
        'recall': r,
        'f1': f1_score(y_valid, pred_thr, zero_division=0),
    })

eligible = [c for c in threshold_candidates if c['precision'] >= TARGET_MIN_PRECISION]
if eligible:
    selected = max(eligible, key=lambda c: c['recall'])
    threshold_strategy = 'max_recall_at_precision_ge_0.80'
    threshold_fallback_used = False
else:
    selected = max(threshold_candidates, key=lambda c: c['f1'])
    threshold_strategy = 'max_f1_fallback'
    threshold_fallback_used = True

final_threshold = selected['threshold']
validation_pred = (validation_proba >= final_threshold).astype(int)

# Test используется только для финальной оценки с уже выбранным threshold.
test_proba = final_model.predict_proba(test_pool)[:, 1]
test_pred = (test_proba >= final_threshold).astype(int)

print('Selected threshold:', round(final_threshold, 3))
print('Threshold strategy:', threshold_strategy)
print('Test ROC AUC:', roc_auc_score(y_test, test_proba))
print('Test Accuracy:', accuracy_score(y_test, test_pred))
print('Test Precision:', precision_score(y_test, test_pred, zero_division=0))
print('Test Recall:', recall_score(y_test, test_pred, zero_division=0))
print('Test F1:', f1_score(y_test, test_pred, zero_division=0))


Selected threshold: 0.5
Threshold strategy: max_recall_at_precision_ge_0.80
Test ROC AUC: 0.8048741418764302
Test Accuracy: 0.7311475409836066
Test Precision: 0.8506493506493507
Test Recall: 0.6894736842105263
Test F1: 0.7616279069767442


## Визуализация результатов (CatBoost)
Графики ключевых метрик, ROC и матрицы ошибок для понятной интерпретации.


In [11]:
import plotly.express as px
import plotly.graph_objects as go
# Метрики импортированы выше

In [12]:
# Гистограмма вероятностей (насколько модель уверена)
fig_p = px.histogram(x=test_proba, nbins=30, title='Распределение предсказанных вероятностей на test')
fig_p.update_layout(xaxis_title='P(класс=1)', yaxis_title='Количество')
fig_p.show()

In [13]:
# Основные метрики
metrics = {
    'AUC': roc_auc_score(y_test, test_proba),
    'Accuracy': accuracy_score(y_test, test_pred),
    'Precision': precision_score(y_test, test_pred, zero_division=0),
    'Recall': recall_score(y_test, test_pred, zero_division=0),
    'F1': f1_score(y_test, test_pred, zero_division=0)
}
metrics_df = pd.DataFrame({'metric': list(metrics.keys()), 'value': list(metrics.values())})
fig_m = px.bar(metrics_df, x='metric', y='value', text='value', title='Ключевые метрики CatBoost')
fig_m.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig_m.update_layout(yaxis=dict(range=[0, 1]))
fig_m.show()

In [14]:
# ROC-кривая
fpr, tpr, _ = roc_curve(y_test, test_proba)
fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name='ROC'))
fig_roc.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Random', line=dict(dash='dash')))
fig_roc.update_layout(title='ROC-кривая (CatBoost)', xaxis_title='FPR', yaxis_title='TPR')
fig_roc.show()

In [15]:
# Матрица ошибок
cm = confusion_matrix(y_test, test_pred)
fig_cm = px.imshow(cm, text_auto=True, aspect='auto',
                   labels=dict(x='Predicted', y='Actual', color='Count'),
                   title='Матрица ошибок (CatBoost)')
fig_cm.show()

## SHAP-анализ (интерактивный Plotly)
Ниже интерактивные визуализации вклада признаков для финальной модели CatBoost на обучающей выборке.


In [16]:
# SHAP values для CatBoost + интерактивные графики Plotly
from plotly.subplots import make_subplots

# 1) SHAP матрица: (n_samples, n_features + 1), последний столбец — expected value
shap_raw = final_model.get_feature_importance(train_pool, type='ShapValues')
shap_values = shap_raw[:, :-1]
expected_value = shap_raw[:, -1]

feature_names = X_train.columns.tolist()
shap_df = pd.DataFrame(shap_values, columns=feature_names, index=X_train.index)

# 2) Глобальная важность (mean |SHAP|)
global_shap = shap_df.abs().mean().sort_values(ascending=False)
top_n = min(12, len(global_shap))
top_features = global_shap.head(top_n).index.tolist()

fig_shap_bar = px.bar(
    global_shap.head(top_n).sort_values(),
    orientation='h',
    labels={'value': 'mean(|SHAP|)', 'index': 'Признак'},
    title='SHAP: глобальная важность признаков (Top-12)',
    color=global_shap.head(top_n).sort_values().values,
    color_continuous_scale='Tealgrn'
)
fig_shap_bar.update_layout(template='plotly_white', coloraxis_showscale=False, height=520)
fig_shap_bar.show()

# 3) Beeswarm-style интерактивный scatter (Top features)
rng = np.random.default_rng(RANDOM_STATE)
sample_size = min(1500, len(X_train))
sample_pos = rng.choice(np.arange(len(X_train)), size=sample_size, replace=False)
Xv = X_train.reset_index(drop=True).iloc[sample_pos].copy()
Sv = shap_df.reset_index(drop=True).iloc[sample_pos].copy()

rows = []
for rank, feat in enumerate(top_features, start=1):
    feat_vals = Xv[feat]
    shap_vals = Sv[feat].values

    if pd.api.types.is_numeric_dtype(feat_vals):
        v = feat_vals.astype(float).values
        vmin, vmax = np.nanmin(v), np.nanmax(v)
        color_norm = (v - vmin) / (vmax - vmin + 1e-9)
        feat_display = feat_vals.round(4).astype(str).values
    else:
        cat_codes = pd.Categorical(feat_vals).codes.astype(float)
        cmax = max(cat_codes.max(), 1.0)
        color_norm = cat_codes / cmax
        feat_display = feat_vals.astype(str).values

    y_base = top_n - rank + 1
    y_jitter = y_base + rng.normal(0, 0.16, size=len(shap_vals))

    rows.append(pd.DataFrame({
        'feature': feat,
        'feature_value': feat_display,
        'shap_value': shap_vals,
        'y': y_jitter,
        'color_norm': color_norm,
    }))

beeswarm_df = pd.concat(rows, ignore_index=True)

fig_bee = go.Figure(
    go.Scattergl(
        x=beeswarm_df['shap_value'],
        y=beeswarm_df['y'],
        mode='markers',
        marker=dict(
            size=6,
            color=beeswarm_df['color_norm'],
            colorscale='RdBu_r',
            cmin=0,
            cmax=1,
            opacity=0.72,
            colorbar=dict(title='Значение фичи (норм.)')
        ),
        customdata=np.stack([beeswarm_df['feature'], beeswarm_df['feature_value']], axis=1),
        hovertemplate='Признак: %{customdata[0]}<br>SHAP: %{x:.4f}<br>Значение: %{customdata[1]}<extra></extra>'
    )
)

fig_bee.update_layout(
    title='SHAP Beeswarm (интерактивный, Top-12)',
    template='plotly_white',
    height=640,
    xaxis_title='SHAP value (вклад в вероятность класса 1)',
    yaxis=dict(
        title='Признаки',
        tickmode='array',
        tickvals=list(range(1, top_n + 1)),
        ticktext=top_features[::-1]
    )
)
fig_bee.show()

# 4) Dependence plot с dropdown (числовые фичи)
train_proba_for_shap = final_model.predict_proba(train_pool)[:, 1]
numeric_top = [f for f in top_features if pd.api.types.is_numeric_dtype(X_train[f])][:8]
if numeric_top:
    fig_dep = go.Figure()

    for i, feat in enumerate(numeric_top):
        fig_dep.add_trace(
            go.Scattergl(
                x=X_train[feat],
                y=shap_df[feat],
                mode='markers',
                marker=dict(
                    size=7,
                    color=train_proba_for_shap,
                    colorscale='Viridis',
                    opacity=0.7,
                    colorbar=dict(title='P(class=1)') if i == 0 else None,
                    showscale=True if i == 0 else False
                ),
                name=feat,
                visible=(i == 0),
                hovertemplate=f'Признак: {feat}<br>Значение: %{{x}}<br>SHAP: %{{y:.4f}}<br>P(1): %{{marker.color:.3f}}<extra></extra>'
            )
        )

    buttons = []
    for i, feat in enumerate(numeric_top):
        vis = [False] * len(numeric_top)
        vis[i] = True
        buttons.append(dict(
            label=feat,
            method='update',
            args=[
                {'visible': vis},
                {'title': f'SHAP Dependence: {feat}', 'xaxis': {'title': feat}}
            ]
        ))

    fig_dep.update_layout(
        template='plotly_white',
        height=560,
        title=f'SHAP Dependence: {numeric_top[0]}',
        xaxis_title=numeric_top[0],
        yaxis_title='SHAP value',
        updatemenus=[dict(buttons=buttons, direction='down', x=1.02, y=1.0, xanchor='left', yanchor='top')]
    )
    fig_dep.show()
else:
    print('Для dependence plot не найдено числовых фич в Top-важных признаках.')

# 5) Таблица Top SHAP для отчета
display(global_shap.head(20).to_frame('mean_abs_shap').round(6))
print('SHAP expected value (mean):', float(np.mean(expected_value)))

,mean_abs_shap
Тип_школы,0.034675
Посещаемость_%,0.030134
Уверенность_в_себе,0.018047
Индекс_качества_школы,0.015199
Доступ_к_ресурсам,0.014006
Пропущенные_дни,0.011343
Уровень_стресса_перед_экзаменом,0.009199
Район,0.004215
Часы_самоподготовки_в_неделю,0.001209
Класс,0.000000


SHAP expected value (mean): 0.012381641591522429


## Сохранение метрик для сравнения


In [17]:
import json
test_tn, test_fp, test_fn, test_tp = confusion_matrix(y_test, test_pred).ravel()

# Validation описывает выбор threshold; test содержит только финальную оценку.
cat_metrics = {
    'model': 'CatBoost',
    'feature_scenario': 'common_no_avg_grade',
    'feature_columns': FEATURE_COLUMNS,
    'excluded_leakage_risk_cols': LEAKAGE_RISK_COLS,
    'split_strategy': 'stratified_train_valid_test',
    'random_state': RANDOM_STATE,
    'train_size': len(X_train),
    'validation_size': len(X_valid),
    'test_size': len(X_test),
    'threshold': float(final_threshold),
    'threshold_strategy': threshold_strategy,
    'threshold_fallback_used': threshold_fallback_used,
    'min_precision_constraint': TARGET_MIN_PRECISION,
    'validation_ROC_AUC': float(roc_auc_score(y_valid, validation_proba)),
    'validation_PR_AUC': float(average_precision_score(y_valid, validation_proba)),
    'validation_Accuracy': float(accuracy_score(y_valid, validation_pred)),
    'validation_Precision': float(precision_score(y_valid, validation_pred, zero_division=0)),
    'validation_Recall': float(recall_score(y_valid, validation_pred, zero_division=0)),
    'validation_F1': float(f1_score(y_valid, validation_pred, zero_division=0)),
    'test_ROC_AUC': float(roc_auc_score(y_test, test_proba)),
    'test_PR_AUC': float(average_precision_score(y_test, test_proba)),
    'test_LogLoss': float(log_loss(y_test, test_proba)),
    'test_Accuracy': float(accuracy_score(y_test, test_pred)),
    'test_Precision': float(precision_score(y_test, test_pred, zero_division=0)),
    'test_Recall': float(recall_score(y_test, test_pred, zero_division=0)),
    'test_F1': float(f1_score(y_test, test_pred, zero_division=0)),
    'test_TN': int(test_tn),
    'test_FP': int(test_fp),
    'test_FN': int(test_fn),
    'test_TP': int(test_tp),
    'CV_AUC_best': float(best_auc),
    'best_params': best_params,
    'best_iteration': int(final_model.get_best_iteration()),
    'reproducibility': {
        'numpy_rng': f'np.random.default_rng({RANDOM_STATE})',
        'catboost_random_seed': RANDOM_STATE,
        'catboost_cv_partition_random_seed': RANDOM_STATE,
        'catboost_task_type': 'CPU',
        'gpu_used': False,
    },
}

out_path = METRICS_DIR / "catboost_metrics.json"
out_path.write_text(json.dumps(cat_metrics, ensure_ascii=False, indent=2), encoding="utf-8")
print('saved', out_path)


saved /Users/mac/Documents/Portfolio/education-quality-dushanbe/Models/Compare models/catboost_metrics.json
